# From Forecasts to Errors: DataFrames in pandas

The slide "Fed Error Over Time" shows how far off the Fed staff's year-ahead forecasts of real
consumption growth were, quarter by quarter, and the trend in those errors. This notebook builds that
result from the raw spreadsheets one step at a time, and each step introduces one idea about pandas
DataFrames. The scripts in `code/main/` do the same work; the last sections compare the two.

The notebook reads only `data/raw/` and writes nothing. Run it from the repository's root folder. It
needs pandas, statsmodels and openpyxl.

In [1]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", "Cannot parse header or footer")  # openpyxl, on the Fed's spreadsheets

## A spreadsheet becomes a DataFrame

The Philadelphia Fed publishes the Greenbook forecasts as one Excel workbook with a sheet for each
variable. `pd.ExcelFile` opens the workbook, and `.parse` reads one sheet into a **DataFrame**: a table
whose columns have names and whose rows have labels, called the **index**.

In [2]:
greenbook = pd.ExcelFile("data/raw/GBweb_Row_Format.xlsx")
greenbook.sheet_names

['Documentation', 'UNEMP', 'gPPCE', 'gPPCEX', 'gRPCE']

In [3]:
cons = greenbook.parse("gRPCE")
cons.shape

(323, 16)

In [4]:
cons.head()

,DATE,gRPCEB4,gRPCEB3,gRPCEB2,gRPCEB1,gRPCEF0,gRPCEF1,gRPCEF2,gRPCEF3,gRPCEF4,gRPCEF5,gRPCEF6,gRPCEF7,gRPCEF8,gRPCEF9,GBdate
0,1978.2,NaN,3.0,9.3,-1.0,5.5,2.9,3.0,5.0,3.9,NaN,NaN,NaN,NaN,NaN,19780614
1,1978.3,NaN,NaN,-0.9,6.2,2.6,3.0,4.9,3.8,3.0,2.7,NaN,NaN,NaN,NaN,19780712
2,1978.3,NaN,NaN,-1.4,6.1,2.5,2.9,5.0,3.9,3.1,2.6,NaN,NaN,NaN,NaN,19780809
3,1978.3,NaN,NaN,-1.4,6.5,1.2,3.0,5.0,3.9,3.1,2.7,NaN,NaN,NaN,NaN,19780913
4,1978.4,NaN,-1.4,6.0,1.7,3.0,4.7,3.7,3.2,2.6,NaN,NaN,NaN,NaN,NaN,19781011


Each row is one Greenbook. `DATE` is the quarter it was written in, and `GBdate` the day. `gRPCEF0` is
its forecast of growth in real personal consumption expenditures (PCE), at an annual rate, for that
quarter, and `gRPCEF1` to `gRPCEF3` are the next three quarters. (`B1` to `B4` are its estimates for
past quarters.) The numbers down the left, 0, 1, 2, ..., are the index pandas made up: row positions,
not dates.

## Choosing rows and columns

A list of column names in brackets picks those columns. A condition picks the rows where it is true.
`.loc[rows, columns]` does both at once. The analysis starts in 1982:

In [5]:
cols = ["gRPCEF0", "gRPCEF1", "gRPCEF2", "gRPCEF3"]
cons = cons.loc[cons["DATE"] >= 1982, ["DATE", "GBdate"] + cols]
cons.head()

,DATE,GBdate,gRPCEF0,gRPCEF1,gRPCEF2,gRPCEF3
35,1982.1,19820127,1.4,2.3,5.5,4.3
36,1982.1,19820324,3.5,2.0,5.5,3.8
37,1982.2,19820512,0.8,5.1,3.7,2.9
38,1982.2,19820623,1.4,4.6,4.0,2.0
39,1982.3,19820818,3.3,4.1,2.9,1.8


The script picks these rows with `[35:]`, because row 35 is the first Greenbook of 1982. That works,
but only someone who counted the rows knows why 35; `cons["DATE"] >= 1982` says what it means. The
index now starts at 35: rows keep their labels when other rows are dropped.

## A number is not a date

`DATE` holds numbers such as 1982.1 for the first quarter of 1982. pandas has a type for a quarter, the
`Period`. Turning the numbers into periods, and making them the index, labels every row with its
quarter:

In [6]:
cons.index = pd.PeriodIndex(cons["DATE"].astype(str).str.replace(".", "Q"), freq="Q", name="quarter")
cons.head()

,DATE,GBdate,gRPCEF0,gRPCEF1,gRPCEF2,gRPCEF3
quarter,,,,,,
1982Q1,1982.1,19820127,1.4,2.3,5.5,4.3
1982Q1,1982.1,19820324,3.5,2.0,5.5,3.8
1982Q2,1982.2,19820512,0.8,5.1,3.7,2.9
1982Q2,1982.2,19820623,1.4,4.6,4.0,2.0
1982Q3,1982.3,19820818,3.3,4.1,2.9,1.8


That's safe here because a quarter only runs from .1 to .4. Data that write months the same way have a
trap: October of 1960, 1960.10, is stored as the number 1960.1, which reads as January. Check a
conversion like this by hand on a few rows.

## How many Greenbooks in a quarter?

The Fed's staff wrote a Greenbook before each meeting of the FOMC, usually two a quarter.
`value_counts` counts how often each value appears; applied to its own result, it counts the counts:

In [7]:
per_quarter = cons.index.value_counts().sort_index()
per_quarter.value_counts()

count
2    134
1      5
3      5
Name: count, dtype: int64

In [8]:
per_quarter[per_quarter != 2]

quarter
1983Q2    1
1983Q3    3
1984Q2    1
1984Q3    3
1985Q2    1
1985Q3    3
1986Q2    1
1986Q3    3
1987Q2    1
1987Q3    3
Freq: Q-DEC, Name: count, dtype: int64

From 1983 to 1987, the second quarter had one Greenbook and the third had three. In 1983:

In [9]:
cons.loc["1983Q2":"1983Q3"]

,DATE,GBdate,gRPCEF0,gRPCEF1,gRPCEF2,gRPCEF3
quarter,,,,,,
1983Q2,1983.2,19830518,3.1,4.7,4.1,3.3
1983Q3,1983.3,19830706,6.0,5.1,3.1,3.2
1983Q3,1983.3,19830817,5.7,4.4,3.3,2.5
1983Q3,1983.3,19830928,3.0,5.0,2.7,2.5


## Keeping the later Greenbook in each quarter

The analysis uses the later Greenbook of each quarter, which is more likely to come after the SPF's
survey. The script takes every other row: `iloc` picks rows by position, and `1::2` means start at the
second row and take every second one after it. That is right only if every quarter has exactly two
Greenbooks:

In [10]:
every_other = cons.iloc[1::2]
every_other.index.value_counts().value_counts()

count
1    134
2      5
Name: count, dtype: int64

Five quarters come out twice, so five others are missing. Saying what we mean instead: keep the last
row of each quarter, by dropping every row whose quarter appears again further down.

In [11]:
last = cons[~cons.index.duplicated(keep="last")]
len(last), last.index.is_unique

(144, True)

In [12]:
wrong = every_other.index != last.index
pd.DataFrame({"the slot for": last.index[wrong],
              "every other row gives the Greenbook of": every_other["GBdate"][wrong].values})

,the slot for,every other row gives the Greenbook of
0,1983Q2,19830706
1,1984Q2,19840711
2,1985Q2,19850703
3,1986Q2,19860702
4,1987Q2,19870701


In each of those five years, the second quarter's slot gets July's Greenbook, which forecasts a
different four quarters. The script then numbers the rows with a fresh list of quarters, so these rows
are labeled as the second quarter. Nothing complains: by position, everything lines up.

## One forecast per row: averaging across columns

The year-ahead forecast is the mean of the four quarterly forecasts. `mean()` averages down each
column (`axis=0`, the default); `axis=1` averages across the columns of each row:

In [13]:
forecast = last[cols].mean(axis=1)
forecast.head()

quarter
1982Q1    3.700
1982Q2    3.000
1982Q3    2.400
1982Q4    2.075
1983Q1    3.075
Freq: Q-DEC, dtype: float64

The result is a **Series**: a single column, with the same index as the DataFrame it came from.

## What happened: rolling means and shifts

The outcomes come from FRED. `index_col` makes the dates the index, and `to_period` turns them into
quarters, the same kind of index as the forecasts have:

In [14]:
fred = pd.read_csv("data/raw/FRED_snapshot.csv", index_col="date", parse_dates=True)
fred.index = fred.index.to_period("Q")
fred.head()

,Unemployment,real Consumption Growth
date,,
1981Q4,8.233333,NaN
1982Q1,8.833333,2.979224
1982Q2,9.433333,1.224315
1982Q3,9.900000,2.711790
1982Q4,10.666667,7.186115


`rolling(4).mean()` gives, at each quarter, the mean of that quarter and the three before it. The
forecast made in quarter *t* is about quarters *t* to *t*+3, so its outcome is the rolling mean at
*t*+3. `shift(-3)` moves each value three quarters earlier, next to the forecast it belongs with:

In [15]:
actual = fred["real Consumption Growth"].rolling(4).mean().shift(-3)
actual.loc["1982Q1"], fred.loc["1982Q1":"1982Q4", "real Consumption Growth"].mean()

(np.float64(3.5253610151676162), np.float64(3.5253610151676162))

## Lining up by date

When two Series meet in arithmetic, pandas lines them up by their index labels, not by position.
Because both are indexed by quarter, `forecast - actual` compares each forecast with the outcome for
the same quarter. A quarter that only one of them has gives `NaN`, "not a number", which `dropna`
removes:

In [16]:
errors = pd.DataFrame({"forecast": forecast, "actual": actual})
errors["error"] = (errors["forecast"] - errors["actual"]).abs()
errors = errors.dropna()
errors.head()

,forecast,actual,error
1982Q1,3.700,3.525361,0.174639
1982Q2,3.000,3.783785,0.783785
1982Q3,2.400,5.576637,3.176637
1982Q4,2.075,6.742821,4.667821
1983Q1,3.075,6.575911,3.500911


To see why the labels matter, drop the first quarter of `actual`. By label, 1982Q1 has nothing to
match and the rest still line up. By position, which is what `reset_index(drop=True)` leaves, every
forecast is compared with the outcome a quarter later, and nothing warns you:

In [17]:
short = actual.loc["1982Q2":]
by_label = (forecast - short).abs()
by_position = (forecast.reset_index(drop=True) - short.reset_index(drop=True)).abs()
pd.DataFrame({"by label": by_label.loc[:"1982Q4"].values, "by position": by_position.iloc[:4].values},
             index=forecast.index[:4])

,by label,by position
quarter,,
1982Q1,NaN,0.083785
1982Q2,0.783785,2.576637
1982Q3,3.176637,4.342821
1982Q4,4.667821,4.500911


## Checking against the scripts

The scripts' errors are in `data/output/abs_errors.csv`. Its rows are the same quarters in the same
order, dated four quarters later, so they can be set side by side with this notebook's:

In [18]:
scripts = pd.read_csv("data/output/abs_errors.csv", index_col=0)["GB_error_cons"]
scripts.index = errors.index
differ = ~np.isclose(errors["error"], scripts)
pd.DataFrame({"this notebook": errors["error"], "the scripts": scripts})[differ].round(2)

,this notebook,the scripts
1983Q2,2.62,2.07
1984Q2,2.07,1.94
1985Q2,1.31,1.89
1986Q2,0.61,1.16
1987Q2,2.85,3.08


The two agree in every quarter except the five where every other row picked July's Greenbook.

## Summaries and groups

`describe` gives the count, mean, spread and quartiles of each column. `groupby` splits the rows into
groups and summarizes each one; here, the mean error in each year:

In [19]:
errors.describe()

,forecast,actual,error
count,144.000000,144.000000,144.000000
mean,2.658160,3.085257,1.016883
std,0.986025,1.605799,0.874796
min,-0.825000,-2.390228,0.002422
25%,2.243750,2.413659,0.314655
50%,2.675000,3.162258,0.743408
75%,3.262500,3.943162,1.520711
max,4.600000,6.742821,4.667821


In [20]:
errors["error"].groupby(errors.index.year).mean().head(8)

1982    2.200720
1983    2.185193
1984    1.763089
1985    1.705287
1986    1.437359
1987    2.479764
1988    1.252970
1989    0.387609
Name: error, dtype: float64

## The trend: a regression

Did the errors get smaller? Regress the error on the year. `HC3` makes the standard errors robust to
errors that are bigger in some years than others. The scripts fit the same regression with `sm.OLS`.

In [21]:
errors["year"] = errors.index.year
fit = smf.ols("error ~ year", data=errors).fit(cov_type="HC3")
fit.summary().tables[1]

,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,78.5359,14.529,5.405,0.000,50.059,107.013
year,-0.0388,0.007,-5.348,0.000,-0.053,-0.025


In [22]:
scripts_fit = smf.ols("error ~ year", data=errors.assign(error=scripts)).fit(cov_type="HC3")
pd.DataFrame({"this notebook": fit.params, "the scripts": scripts_fit.params}).loc[["year"]].round(4)

,this notebook,the scripts
year,-0.0388,-0.0393


The error shrank by about 0.04 percentage points a year, about 1.4 points between 1982 and 2017. The
five misplaced Greenbooks barely move it.

## Try it yourself

- **Unemployment.** In the `UNEMP` sheet, the forecast of the change over the year is
  `UNEMPF4 - UNEMPF0`. The outcome is `fred["Unemployment"].diff(4)`, the change over the four quarters
  up to each date, so it needs `shift(-4)`. Is there a trend in these errors?
- **The SPF.** `data/raw/meanGrowth.xlsx`, sheet `RCONSUM`, has `YEAR` and `QUARTER` columns instead of
  one `DATE`. Build its index with `pd.PeriodIndex.from_fields(year=..., quarter=..., freq="Q")`, then
  repeat the steps with `drconsum2` to `drconsum5`.